# Sentiment Analysis 💬
**Problem:** Read a short review and decide if it is **positive** or **negative**.
**Type:** Text classification (NLP) · **Dataset:** 3,000 labelled sentences from IMDb, Yelp and Amazon (UCI Sentiment Labelled Sentences) — 2,748 usable after removing rows corrupted by stray tab characters

## 1. Load the data

In [1]:
import pandas as pd, io, zipfile, urllib.request

url = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
raw = urllib.request.urlopen(url).read()
z = zipfile.ZipFile(io.BytesIO(raw))
frames = []
for name in z.namelist():
    if name.endswith("_labelled.txt"):
        part = pd.read_csv(io.BytesIO(z.read(name)), sep="\t", header=None, names=["text", "label"], encoding="latin-1")
        frames.append(part)
df = pd.concat(frames).reset_index(drop=True)
# Some sentences contain stray tabs, which can shift a row's columns.
# Keep only rows whose label is a clean 0 or 1.
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label", "text"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)
print(df.shape)
print(df["label"].value_counts())
df.head()

(2748, 2)
label
1    1386
0    1362
Name: count, dtype: int64


,text,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


## 2. Turn text into numbers with TF-IDF
TF-IDF scores each word by how *distinctive* it is — common words like "the" count for little, words like "terrible" or "excellent" count for a lot. I use 1–2 word phrases (so "not good" is captured as one feature).

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
Xtr = vectorizer.fit_transform(X_train)
Xte = vectorizer.transform(X_test)
print("Feature matrix:", Xtr.shape)

Feature matrix: (2198, 5000)


## 3. Train and evaluate

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(Xtr, y_train)
pred = model.predict(Xte)
print(f"Accuracy: {accuracy_score(y_test, pred):.3f}")
print(classification_report(y_test, pred, target_names=["negative", "positive"]))

Accuracy: 0.822
              precision    recall  f1-score   support

    negative       0.79      0.87      0.83       273
    positive       0.86      0.78      0.81       277

    accuracy                           0.82       550
   macro avg       0.82      0.82      0.82       550
weighted avg       0.82      0.82      0.82       550



## 4. Try it on new sentences

In [4]:
tests = [
    "This product is amazing, I absolutely love it!",
    "Worst experience ever, the food was cold and the staff was rude.",
    "The movie was brilliant, great acting and a beautiful story.",
    "Total waste of money, I will never buy this again.",
]
for text in tests:
    p = model.predict(vectorizer.transform([text]))[0]
    print(("POSITIVE" if p == 1 else "NEGATIVE"), "<-", text)

POSITIVE <- This product is amazing, I absolutely love it!
NEGATIVE <- Worst experience ever, the food was cold and the staff was rude.
POSITIVE <- The movie was brilliant, great acting and a beautiful story.
NEGATIVE <- Total waste of money, I will never buy this again.


## What I learned
- TF-IDF + Logistic Regression is a strong, fast baseline for sentiment — no deep learning needed for clean, short reviews.
- Bigrams matter: "not good" and "good" must not look the same to the model.
- The same workflow powers real business tools: review monitoring, customer-feedback dashboards and brand-sentiment tracking.